In [4]:
import os

# Directory where embeddings are saved
EMB_DIR = "../embeddings"


In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torchvision.transforms as T
from torchvision import models

from sklearn.decomposition import PCA
from joblib import dump

# Pick device: MPS for Mac M1/M2, else CUDA, else CPU
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", device)

# Paths
TRAIN_CSV = "../dataset/train.csv"
TEST_CSV  = "../dataset/test.csv"
IMG_DIR_TRAIN = "../images/train"
IMG_DIR_TEST  = "../images/test"
EMB_DIR       = "../embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)


Torch device: mps


In [2]:
# Preprocessing pipeline for images
transform = T.Compose([
    T.Resize((224, 224)),  # standard ResNet input size
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load pretrained ResNet50
model = models.resnet50(pretrained=True)
model.fc = torch.nn.Identity()  # remove classification head, keep features
model = model.to(device)
model.eval()


/opt/anaconda3/envs/student_resource/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/student_resource/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/dosvatsky/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:06<00:00, 15.9MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [3]:
def get_embedding(img_path):
    try:
        img = Image.open(img_path).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = model(x).cpu().numpy().flatten()
        return feat
    except:
        return np.zeros(2048)  # fallback for missing/broken images


In [5]:
def extract_embeddings(df, img_dir, save_path):
    feats, ids = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        sid = int(row["sample_id"])
        img_path = os.path.join(img_dir, f"{sid}.jpg")
        vec = get_embedding(img_path)
        feats.append(vec)
        ids.append(sid)
    feats = np.array(feats)
    np.save(save_path, feats)
    return ids, feats


In [6]:
# Smoke test: only 100 images from train
train_ids_small, train_feats_small = extract_embeddings(
    train.sample(100, random_state=42), 
    IMG_DIR_TRAIN, 
    os.path.join(EMB_DIR, "train_resnet_small.npy")
)

print("Shape of embeddings:", train_feats_small.shape)


100%|██████████| 100/100 [00:05<00:00, 17.75it/s]

Shape of embeddings: (100, 2048)


In [7]:
# Full extraction for train
train_ids, train_feats = extract_embeddings(
    train, 
    IMG_DIR_TRAIN, 
    os.path.join(EMB_DIR, "train_resnet.npy")
)

# Full extraction for test
test_ids, test_feats = extract_embeddings(
    test,  
    IMG_DIR_TEST,  
    os.path.join(EMB_DIR, "test_resnet.npy")
)

print("Train embeddings:", train_feats.shape)
print("Test embeddings:", test_feats.shape)


100%|██████████| 75000/75000 [51:38<00:00, 24.20it/s]  


Train embeddings: (75000, 2048)
Test embeddings: (75000, 2048)


In [6]:
from sklearn.decomposition import PCA
import numpy as np
import joblib

# Load saved embeddings
train_feats = np.load(os.path.join(EMB_DIR, "train_resnet.npy"))
test_feats  = np.load(os.path.join(EMB_DIR, "test_resnet.npy"))

# PCA to 256 dims
pca = PCA(n_components=256, random_state=42)
train_feats_pca = pca.fit_transform(train_feats)
test_feats_pca  = pca.transform(test_feats)

print("After PCA:", train_feats_pca.shape, test_feats_pca.shape)

# Save for reuse
np.save(os.path.join(EMB_DIR, "train_resnet_pca.npy"), train_feats_pca)
np.save(os.path.join(EMB_DIR, "test_resnet_pca.npy"), test_feats_pca)
joblib.dump(pca, os.path.join(EMB_DIR, "pca_model.pkl"))


After PCA: (75000, 256) (75000, 256)


['../embeddings/pca_model.pkl']

In [7]:
import numpy as np
import os

# Make sure features folder exists
os.makedirs("../features", exist_ok=True)

# Save PCA-reduced image embeddings
np.save("../features/train_img.npy", train_feats_pca)
np.save("../features/test_img.npy", test_feats_pca)

print("Saved:")
print("- ../features/train_img.npy", train_feats_pca.shape)
print("- ../features/test_img.npy", test_feats_pca.shape)


Saved:
- ../features/train_img.npy (75000, 256)
- ../features/test_img.npy (75000, 256)


In [5]:
import os

print("Contents of EMB_DIR:", os.listdir(EMB_DIR))


Contents of EMB_DIR: ['train_resnet_small.npy', 'train_resnet.npy', 'test_resnet.npy', 'test_resnet_pca.npy', 'train_resnet_pca.npy', 'pca_model.pkl']
